# 01 — Data Audit

**Purpose:** Verify the exact state of the frozen dataset.  
Reproduce verified findings: 48 completed matches, group standings through MD2,  
third-place ranking, name mapping correctness, and data quality summary.

**Freeze date:** 2026-06-23 23:59 UTC  
**Elo snapshot:** 2026-05-27  
**This notebook produces no model inputs — only verification evidence.**

---
**Outputs**
- `outputs/group_standings_freeze.csv`

In [1]:
import sys
from pathlib import Path


PROJECT_ROOT = Path("__file__").resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import hashlib
import pandas as pd

from src.leakage_guard import (
    FREEZE_DATE, ELO_SNAPSHOT_DATE,
    check_freeze_date, check_elo_snapshot, check_canonical_names,
    LeakageError,
)
from src.name_map import CANONICAL_48, TO_CANONICAL, canonicalize, assert_all_canonical
from src.standings import build_group_membership, compute_group_standings, rank_third_placed
from src.features import (
    load_ds2, load_ds4, load_ds6, load_ds8, load_ds10,
    load_ds16, load_ds17, load_ds18, load_ds19,
    load_ds1, load_ds1ext,
)

DATA_ROOT   = PROJECT_ROOT
ARC_BASE    = DATA_ROOT / "archive.zip"          # DS16, DS17, DS18, DS19
ARC2        = DATA_ROOT / "archive (2).zip"      # DS8, DS10, DS11
ARC3        = DATA_ROOT / "archive (3).zip"      # DS1 (2026 WC results)
ARC4        = DATA_ROOT / "archive (4).zip"      # DS2 (Elo)
ARC6        = DATA_ROOT / "archive (6).zip"      # DS4, DS6
JUNE23      = DATA_ROOT / "june23_results.csv"   # DS1-ext: 4 June-23 matches
OUTPUTS     = PROJECT_ROOT / "outputs"
OUTPUTS.mkdir(exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Freeze date  : {FREEZE_DATE}")
print(f"Elo snapshot : {ELO_SNAPSHOT_DATE}")

Project root : /Users/pranaavdhaksheshganesh/Downloads/FIFA_WC_2026_Project
Freeze date  : 2026-06-23
Elo snapshot : 2026-05-27


## Cell 2 — Load raw datasets

In [2]:
ds2  = load_ds2(ARC4)       # Elo ratings snapshot
ds4  = load_ds4(ARC6)       # International match results
ds6  = load_ds6(ARC6)       # Penalty shootout records
ds8  = load_ds8(ARC2)       # Historical World Cup matches (1930–2022)
ds10 = load_ds10(ARC2)      # FIFA rankings
ds16 = load_ds16(ARC_BASE)  # 2026 bracket fixtures
ds17 = load_ds17(ARC_BASE)  # 2026 teams / group assignments
ds1  = load_ds1(ARC3)       # 2026 completed WC matches (44 rows, through June 22)
ds1ext = load_ds1ext(JUNE23)  # 4 June-23 matches (score-only, no tactical columns)

# Merge into frozen 48-match result set
completed_48 = pd.concat([ds1, ds1ext], ignore_index=True)

# Freeze-date and Elo snapshot leakage guards
ds4_frozen = ds4[ds4["date"] <= pd.Timestamp("2026-06-23")].copy()
ds2_frozen = ds2[
    (ds2["year"] == 2026) &
    (ds2["snapshot_date"] == pd.Timestamp("2026-05-27"))
].copy()

check_freeze_date(ds1,        date_column="date")
check_freeze_date(ds1ext,     date_column="date")
check_freeze_date(ds4_frozen, date_column="date")
check_elo_snapshot(ds2_frozen)

print("✓ Raw datasets loaded successfully")
print()
print(f"DS1        : {ds1.shape}   (2026 results through June 22)")
print(f"DS1-ext    : {ds1ext.shape}   (June 23 matches)")
print(f"completed_48: {completed_48.shape}  (frozen 48-match set)")
print(f"DS2        : {ds2.shape}   (Elo — all snapshots)")
print(f"DS4        : {ds4.shape}   (international results)")
print(f"DS4*       : {ds4_frozen.shape}   (freeze-filtered)")
print(f"DS6        : {ds6.shape}   (shootouts)")
print(f"DS8        : {ds8.shape}   (historical World Cup archive)")
print(f"DS10       : {ds10.shape}  (FIFA rankings)")
print(f"DS16       : {ds16.shape}  (2026 bracket)")
print(f"DS17       : {ds17.shape}  (2026 teams)")

✓ Raw datasets loaded successfully

DS1        : (44, 42)   (2026 results through June 22)
DS1-ext    : (4, 7)   (June 23 matches)
completed_48: (48, 43)  (frozen 48-match set)
DS2        : (4683, 23)   (Elo — all snapshots)
DS4        : (49477, 9)   (international results)
DS4*       : (49453, 9)   (freeze-filtered)
DS6        : (678, 5)   (shootouts)
DS8        : (964, 46)   (historical World Cup archive)
DS10       : (211, 9)  (FIFA rankings)
DS16       : (104, 8)  (2026 bracket)
DS17       : (48, 6)  (2026 teams)


/Users/pranaavdhaksheshganesh/Downloads/FIFA_WC_2026_Project/src/features.py:214: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df["kickoff_at"]    = pd.to_datetime(df["kickoff_at"], errors="coerce", utc=False)


## Cell 3 — Freeze-date and Elo snapshot leakage checks

In [3]:

check_freeze_date(ds1,        date_column="date")
check_freeze_date(ds1ext,     date_column="date")
check_freeze_date(ds4_frozen, date_column="date")
check_elo_snapshot(ds2_frozen)

print("✓ Freeze-date check passed — no post-freeze data in DS1, DS1-ext, or DS4*")
print("✓ Elo snapshot check passed — snapshot date is 2026-05-27")

✓ Freeze-date check passed — no post-freeze data in DS1, DS1-ext, or DS4*
✓ Elo snapshot check passed — snapshot date is 2026-05-27


## Cell 4 — Frozen 48-match result set

In [4]:
# completed_48 = DS1 (44 rows, through June 22) + DS1-ext (4 June-23 rows)
completed = completed_48.dropna(subset=["home_score", "away_score"]).copy()
completed["home_score"] = completed["home_score"].astype(int)
completed["away_score"] = completed["away_score"].astype(int)

assert len(completed) == 48, (
    f"Expected exactly 48 completed matches, got {len(completed)}"
)
print(f"✓ Exactly 48 completed matches confirmed  (DS1: {len(ds1)}, DS1-ext: {len(ds1ext)})")

# Display the full 48-match table
display_cols = ["date", "home_team", "away_team", "home_score", "away_score"]
available = [c for c in display_cols if c in completed.columns]
completed[available].sort_values("date").reset_index(drop=True)

✓ Exactly 48 completed matches confirmed  (DS1: 44, DS1-ext: 4)


,date,home_team,away_team,home_score,away_score
0,2026-06-11,Mexico,South Africa,2,0
1,2026-06-11,Korea Republic,Czechia,2,1
2,2026-06-12,Canada,Bosnia-Herzegovina,1,1
3,2026-06-12,United States,Paraguay,4,1
4,2026-06-13,Qatar,Switzerland,1,1
5,2026-06-13,Brazil,Morocco,1,1
6,2026-06-13,Haiti,Scotland,0,1
7,2026-06-13,Australia,Türkiye,2,0
8,2026-06-14,Sweden,Tunisia,5,1
9,2026-06-14,Côte d'Ivoire,Ecuador,1,0


## Cell 5 — Compute and display group standings

In [5]:
group_membership = build_group_membership(ds17)

# FIFA ranks for tiebreaker (optional — pass None to use fallback)
fifa_ranks = None
if "fifa_points" in ds10.columns and "country" in ds10.columns:
    fifa_ranks = dict(zip(ds10["country"].map(canonicalize), ds10["fifa_points"]))

standings = compute_group_standings(completed, group_membership, fifa_ranks=fifa_ranks)

print(f"Standings shape: {standings.shape}  (expect 48 rows × 11 cols)")

# Spot-check verified values from the design spec
def _pts(team):
    row = standings[standings["team"] == team]
    assert len(row) == 1, f"{team} not found in standings"
    return int(row["pts"].iloc[0])

for team, expected_pts in [("Mexico", 6), ("France", 6), ("Norway", 6),
                            ("Germany", 6), ("Colombia", 6)]:
    try:
        actual = _pts(team)
        status = "✓" if actual == expected_pts else "✗"
        print(f"{status} {team}: {actual} pts (expected {expected_pts})")
    except AssertionError as e:
        print(f"  SKIP: {e}")

standings

Standings shape: (48, 11)  (expect 48 rows × 11 cols)
✓ Mexico: 6 pts (expected 6)
✓ France: 6 pts (expected 6)
✓ Norway: 6 pts (expected 6)
✓ Germany: 6 pts (expected 6)
✓ Colombia: 6 pts (expected 6)


,team,group,pts,w,d,l,gf,ga,gd,matches_played,group_rank
0,Mexico,A,6,2,0,0,3,0,3,2,1
1,Korea Republic,A,3,1,0,1,2,2,0,2,2
2,Czechia,A,1,0,1,1,2,3,-1,2,3
3,South Africa,A,1,0,1,1,1,3,-2,2,4
4,Canada,B,4,1,1,0,7,1,6,2,1
5,Switzerland,B,4,1,1,0,5,2,3,2,2
6,Bosnia-Herzegovina,B,1,0,1,1,2,5,-3,2,3
7,Qatar,B,1,0,1,1,1,7,-6,2,4
8,Brazil,C,4,1,1,0,4,1,3,2,1
9,Morocco,C,4,1,1,0,2,1,1,2,2


## Cell 6 — Third-place ranking

In [9]:
ranked_thirds = rank_third_placed(standings, fifa_ranks=fifa_ranks)

assert len(ranked_thirds) == 12, f"Expected 12 third-placed teams, got {len(ranked_thirds)}"
assert ranked_thirds["qualifies_as_third"].sum() == 8, "Exactly 8 thirds should qualify"

print("Third-place ranking (top 8 qualify for R32):")
display_cols = ["team", "group", "pts", "gd", "gf", "third_place_rank", "qualifies_as_third"]
available = [c for c in display_cols if c in ranked_thirds.columns]
ranked_thirds[available].sort_values("third_place_rank").reset_index(drop=True)

Third-place ranking (top 8 qualify for R32):


,team,group,pts,gd,gf,third_place_rank,qualifies_as_third
0,Sweden,F,3,0,6,1,True
1,Scotland,C,3,0,1,2,True
2,Croatia,L,3,-1,3,3,True
3,Paraguay,D,3,-2,2,4,True
4,Algeria,J,3,-2,2,5,True
5,Cape Verde,H,2,0,2,6,True
6,Belgium,G,2,0,1,7,True
7,Czechia,A,1,-1,2,8,True
8,Congo DR,K,1,-1,1,9,False
9,Ecuador,E,1,-1,0,10,False


## Cell 7 — Name mapping validation

In [10]:
assert len(CANONICAL_48) == 48, f"Expected 48 canonical teams, got {len(CANONICAL_48)}"

# Verify every canonical name round-trips through canonicalize()
failures = [t for t in CANONICAL_48 if canonicalize(t) != t]
assert not failures, f"Canonical names failed round-trip: {failures}"

# Spot-check known variant mappings
known_variants = {
    "South Korea":           "Korea Republic",
    "Iran":                  "IR Iran",
    "Ivory Coast":           "Côte d'Ivoire",
    "DR Congo":              "Congo DR",
    "Turkey":                "Türkiye",
    "USA":                   "United States",
    "Czech Republic":        "Czechia",
    "Bosnia and Herzegovina":"Bosnia-Herzegovina",
}
variant_fails = {v: canonicalize(v) for v, expected in known_variants.items()
                 if canonicalize(v) != expected}
assert not variant_fails, f"Variant mapping failures: {variant_fails}"

print(f"✓ {len(CANONICAL_48)} canonical teams verified")
print(f"✓ {len(TO_CANONICAL)} variant mappings registered")
print(f"✓ All {len(known_variants)} spot-checked variants correct")

# Display full mapping table
mapping_rows = [{"canonical": t, "sample_variant": next(
    (k for k, v in TO_CANONICAL.items() if v == t), "(identity)")
} for t in sorted(CANONICAL_48)]
pd.DataFrame(mapping_rows)

✓ 48 canonical teams verified
✓ 11 variant mappings registered
✓ All 8 spot-checked variants correct


,canonical,sample_variant
0,Algeria,(identity)
1,Argentina,(identity)
2,Australia,(identity)
3,Austria,(identity)
4,Belgium,(identity)
5,Bosnia-Herzegovina,Bosnia and Herzegovina
6,Brazil,(identity)
7,Canada,(identity)
8,Cape Verde,Cabo Verde
9,Colombia,(identity)


## Cell 8 — Data quality summary

In [11]:
quality_rows = []
for name, df in [("DS1 (2026 results)", ds1), ("DS2 (Elo)", ds2),
                 ("DS4 (competitive)", ds4), ("DS6 (shootouts)", ds6),
                 ("DS8 (WC archive)", ds8), ("DS10 (FIFA rank)", ds10)]:
    null_counts = df.isnull().sum()
    cols_with_nulls = null_counts[null_counts > 0]
    quality_rows.append({
        "dataset": name,
        "rows": len(df),
        "cols": len(df.columns),
        "cols_with_nulls": len(cols_with_nulls),
        "total_nulls": int(null_counts.sum()),
        "null_detail": "; ".join(f"{c}:{v}" for c, v in cols_with_nulls.items()) or "none",
    })

quality_df = pd.DataFrame(quality_rows)
print("Data quality summary:")
quality_df

Data quality summary:


,dataset,rows,cols,cols_with_nulls,total_nulls,null_detail
0,DS1 (2026 results),44,42,3,46,home_offsides:1; away_offsides:1; notes:44
1,DS2 (Elo),4683,23,0,0,none
2,DS4 (competitive),49477,9,2,88,home_score:44; away_score:44
3,DS6 (shootouts),678,5,1,422,first_shooter:422
4,DS8 (WC archive),964,46,31,20914,home_xg:836; home_penalty:929; away_xg:836; aw...
5,DS10 (FIFA rank),211,9,0,0,none


## Cell 9 — Save outputs

In [12]:
out_path = OUTPUTS / "group_standings_freeze.csv"
standings.to_csv(out_path, index=False)
print(f"Saved → {out_path}  ({len(standings)} rows)")

# Also save ranked thirds
thirds_path = OUTPUTS / "third_place_ranking_freeze.csv"
ranked_thirds.to_csv(thirds_path, index=False)
print(f"Saved → {thirds_path}  ({len(ranked_thirds)} rows)")

Saved → /Users/pranaavdhaksheshganesh/Downloads/FIFA_WC_2026_Project/outputs/group_standings_freeze.csv  (48 rows)
Saved → /Users/pranaavdhaksheshganesh/Downloads/FIFA_WC_2026_Project/outputs/third_place_ranking_freeze.csv  (12 rows)
